# 응용 모의고사 Set 1 — 정답 — 문자열 추출과 상품 분석

- 데이터: `TV.csv`
- 난이도: 기존 Set 01~06과 유사
- 구성: **공통 전처리 → Q1 통계 → Q2 상관분석 → Q3 모델링**
- 모든 문항은 공통 전처리 결과를 이어서 사용합니다.
- 전처리 완료 후 데이터는 **197행**이어야 합니다. 행 수가 다르면 다음 문제로 넘어가기 전에 전처리를 확인하세요.

정답 노트북은 `../answers/`에 있습니다.

## 공통 전처리 정답

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv('../../dataset/TV.csv')
base = df.copy()
spec_cols = ['Picture_quality', 'Speaker', 'Frequency']
base['spec_text'] = base[spec_cols].fillna('').astype(str).agg(' '.join, axis=1)
base['refresh_rate'] = pd.to_numeric(
    base['spec_text'].str.extract(r'(\d{2,3})\s*Hz', expand=False),
    errors='coerce'
)
exclude = base['channel'].str.contains(r'Pixel|Oper', na=False)
base = base.loc[~exclude].copy()
base['review_ratio'] = base['Reviews'] / base['Ratings']
base['price_ratio'] = base['current_price'] / base['MRP']
base['Netflix'] = base['channel'].str.contains('Netflix', na=False).astype(int)
base['PrimeVideo'] = base['channel'].str.contains('Prime Video', na=False).astype(int)
base['high_quality'] = base['Picture_quality'].str.contains(r'4K|Ultra HD', na=False).astype(int)
base = base.replace([np.inf, -np.inf], np.nan)
base = base.dropna(subset=['refresh_rate', 'review_ratio', 'price_ratio']).copy()
assert len(base) == 197
display(base.head())

## Q1 정답

In [ ]:
high_mean = base.loc[base['refresh_rate'] >= 60, 'Stars'].mean()
low_mean = base.loc[base['refresh_rate'] < 60, 'Stars'].mean()
answer_q1 = round(high_mean - low_mean, 2)
display(answer_q1)  # -0.05

## Q2 정답

In [ ]:
corr = base[['Stars', 'refresh_rate', 'MRP', 'current_price']].corr(method='pearson')
stars_corr = corr['Stars'].drop(index='Stars')
answer_var_q2 = stars_corr.abs().idxmax()
answer_coef_q2 = round(stars_corr.loc[answer_var_q2], 3)
display(answer_var_q2, answer_coef_q2)  # current_price, 0.184

## Q3 정답

In [ ]:
from sklearn.ensemble import RandomForestRegressor

features = ['review_ratio', 'MRP', 'price_ratio', 'Netflix',
            'PrimeVideo', 'high_quality', 'refresh_rate']
X = base[features]
y = base['Stars']
model = RandomForestRegressor(random_state=321)
model.fit(X, y)
importance = pd.Series(model.feature_importances_, index=features)
answer_q3 = importance.idxmax()
display(importance.sort_values(ascending=False), answer_q3)  # price_ratio